In [1]:
import pickle
import math
import numpy as np

with open('Data/mel_spec.pkl', 'rb') as file:
    
    (X_melspec_train, X_melspec_test, X_melspec_valid, y_melspec_train, y_melspec_test, y_melspec_valid) = pickle.load(file)
    #(X_mfcc_train, X_mfcc_test, X_mfcc_valid, y_mfcc_train, y_mfcc_test, y_mfcc_valid) = pickle.load(file)
    #(X_cqt_train, X_cqt_test, X_cqt_valid, y_cqt_train, y_cqt_test, y_cqt_valid) = pickle.load(file)

In [2]:
y_train = []
X_train = []
for i in range(len(X_melspec_train)):
    for j in range(X_melspec_train[i].shape[1]):
        y_train.append(y_melspec_train[i][math.floor(j*len(y_melspec_train[i])/X_melspec_train[i].shape[1])])
        X_train.append(X_melspec_train[i].T[j])
    

In [3]:
y_test = []
X_test = []
for i in range(len(X_melspec_test)):
    for j in range(X_melspec_test[i].shape[1]):
        y_test.append(y_melspec_test[i][math.floor(j*len(y_melspec_test[i])/X_melspec_test[i].shape[1])])
        X_test.append(X_melspec_test[i].T[j])

In [4]:
y_train = np.array(y_train)
X_train = np.array(X_train)
y_test = np.array(y_test)
X_test = np.array(X_test)

In [5]:
X_train.shape, y_train.shape

((1041470, 128), (1041470, 88))

In [10]:
import tensorflow as tf

INFO:tensorflow:Enabling eager execution
INFO:tensorflow:Enabling v2 tensorshape
INFO:tensorflow:Enabling resource variables
INFO:tensorflow:Enabling tensor equality
INFO:tensorflow:Enabling control flow v2


In [12]:
input_shape = X_train[0].shape
print(input_shape)

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(88, activation='softmax'),
])


(128,)


In [13]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_4 (Dense)              (None, 128)               16512     
_________________________________________________________________
dense_5 (Dense)              (None, 128)               16512     
_________________________________________________________________
dense_6 (Dense)              (None, 128)               16512     
_________________________________________________________________
dense_7 (Dense)              (None, 88)                11352     
Total params: 60,888
Trainable params: 60,888
Non-trainable params: 0
_________________________________________________________________


In [14]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [15]:
model.fit(X_train[:65526], y_train[:65526], epochs=2, batch_size = 128)

Epoch 1/2
512/512 [==============================] - 2s 2ms/step - loss: 0.1600 - accuracy: 0.0219: 0s - loss: 0.1731 - accu
Epoch 2/2
512/512 [==============================] - 1s 2ms/step - loss: 0.1041 - accuracy: 0.0615


In [16]:
loss, accuracy = model.evaluate(X_test, y_test)

12078/12078 [==============================] - 8s 611us/step - loss: 0.1207 - accuracy: 0.0618


In [74]:
def get_keys(label):
    keys = []
    for i in range(label.shape[0]):
        if label[i] == 1:
            keys.append(i)
    return keys

def get_highest_scores(pred, threshold):
    keys = []
    for i in range(pred.shape[0]):
        if pred[i] >= threshold:
            keys.append((i, float(pred[i])))
    return keys

In [75]:
def predict(input):
    out = []
    prediction = model(input)
    for vec in prediction:
        out.append(get_highest_scores(vec, 0.1))
    return out

In [94]:
predict([X_test[0:1]])

[[]]

In [93]:
get_keys(y_test[0])

[]